# Lab 13 starter: a display policy and the range of a shadow price

This notebook is standalone. In the recap cell, the product and order tables are
loaded, the chapter's six candidates and their demand caps are rebuilt, and the
chapter's function `build_model` is defined. The original model is built and solved
once, so that you can compare your changed models with it.

The lab prompt is at the end of Chapter 13, under **Exercises**.

**Build lab.** Merchandising adds a policy: no single product may occupy more than
40% of the display volume, that is, more than 100 cubic feet. Add one constraint per
product to the LP and re-solve it. Report the new plan, the new total margin, and
the cost of the policy in dollars of margin. Then report the shadow prices of space
and budget. State in two sentences why the shadow price of space changed in the
direction it did, and why the shadow price of the budget changed.

**Evaluate lab.** Find the range of the budget's shadow price by experiment.
Re-solve the original LP at buying budgets from \$20,000 to \$30,000, in steps of
\$500. Tabulate the optimal margin and the budget's shadow price at each budget, and
compute the margin gained per added dollar between consecutive budgets. Compare it
with the shadow price of \$0.45. State in two sentences the range of budgets over
which the shadow price held, and what you would tell finance about a budget of
\$28,000.

## Running this notebook

Run the setup cell below first, then the recap cell, then work down. In Google
Colab nothing needs to be installed beyond that cell. Locally, use the
`pyba-core` environment from Appendix A.

You will enter your work in the cells marked `# TODO`. Everything else is
provided. When you are unsure what a step produced, print it: `.head()` for a
table or a column, and the bare name for a number.

**If you are in Google Colab:** this notebook opened read-only from GitHub. Click
**Copy to Drive** in the toolbar (or *File > Save a copy in Drive*) before you
edit anything, then work in that copy. It saves to your Google Drive under
*Colab Notebooks*. Edits made to the read-only original are lost when the tab
closes.

In [ ]:
# Setup. Run this cell once per session. It installs this lab's packages;
# on Google Colab it also fetches the course data.
%pip install -q pandas pulp==3.3.2
import sys
if "google.colab" in sys.modules:
    !test -d pyba-companion || git clone --quiet --depth 1 https://github.com/murtaza-nasir/pyba-companion.git
    sys.path.insert(0, "pyba-companion")   # makes `import pyba` (DATA_DIR) work

## Recap: the Chapter 13 setup

The recap cell contains the chapter's code:

- `data` is the table of the six candidates, one row per product, labeled by SKU.
  It has the columns `margin`, `case_cube_ft`, `unit_cost`, and `demand_cap`.
- `build_model(space, budget)` is the chapter's function from the section "Shadow
  price range". It returns the promotion model, before solving, and the
  dictionary `qty` of its six variables. The two constraints are named
  `"display_space"` and `"budget"`.
- `model` is the original LP, solved once. `lp_margin` is its total margin, and
  `space_price` and `budget_price` are its two shadow prices.

The cell prints the original plan and the two shadow prices, so that you can
compare the policy plan with them in Step 5.

In [ ]:
# --- Chapter 13 recap: run this cell first ---
import pandas as pd
import pulp

from pyba import DATA_DIR

products = pd.read_csv(DATA_DIR / "pw_products.csv")
orders = pd.read_csv(DATA_DIR / "pw_orders.csv")
products["margin"] = (products["unit_price"] - products["unit_cost"]).round(2)
candidates = products.nlargest(6, "margin")
units = orders.groupby("sku", as_index=False)["quantity"].sum()
candidates = candidates.merge(units, on="sku", how="left", validate="one_to_one")
candidates["demand_cap"] = (candidates["quantity"] / 36 * 1.5).round().astype(int)
data = candidates.set_index("sku")


def build_model(space, budget, cat="Continuous"):
    model = pulp.LpProblem("fall_promotion", pulp.LpMaximize)
    qty = {}
    for sku in data.index:
        qty[sku] = pulp.LpVariable(sku, lowBound=0, upBound=data.loc[sku, "demand_cap"], cat=cat)
    model += pulp.lpSum([data.loc[sku, "margin"] * qty[sku] for sku in data.index])
    model += pulp.lpSum([data.loc[sku, "case_cube_ft"] * qty[sku] for sku in data.index]) <= space, "display_space"
    model += pulp.lpSum([data.loc[sku, "unit_cost"] * qty[sku] for sku in data.index]) <= budget, "budget"
    return model, qty


model, qty = build_model(250, 25_000)
model.solve(pulp.PULP_CBC_CMD(msg=False))
lp_margin = pulp.value(model.objective)
space_price = model.constraints["display_space"].pi
budget_price = model.constraints["budget"].pi

print(f"original LP: {pulp.LpStatus[model.status]}, margin ${lp_margin:,.2f}")
print(f"shadow prices: space {space_price:.4f}, budget {budget_price:.4f}")
original = data[["product_name", "case_cube_ft"]].copy()
original["cases"] = [qty[sku].varValue for sku in data.index]
original["space_used"] = original["cases"] * original["case_cube_ft"]
original.round(2)

## Step 1: Add the policy constraints

Build a new copy of the LP with `policy, policy_qty = build_model(250, 25_000)`.
Then add one constraint per product: the space it occupies, its `case_cube_ft`
times its variable, is at most `0.4 * 250`. Use a `for` loop over `data.index`, as
in the chapter's loop that creates the variables, and add each constraint with
`+=`, as in the section "Model formulation in PuLP". Give each constraint its own name,
for example `f"share_{sku}"`.

`policy` then has eight constraints: the two of the chapter and six new ones.

In [ ]:
# TODO: build the model again and add one 40% constraint per product
policy, policy_qty = None, None

# --- Look: the number of constraints ---
print(len(policy.constraints), "constraints")

## Step 2: Solve the policy model

Solve `policy` with `pulp.PULP_CBC_CMD(msg=False)` and print its status and total
margin, as in the chapter. Then make the table `policy_plan`: a copy of
`data[["product_name", "case_cube_ft"]]` with a `cases` column, the value of each
variable from `policy_qty`, and a `space_used` column, cases times
`case_cube_ft`.

In the original plan (recap cell), the liners occupy 122.36 cubic feet. Under the
policy, no product may occupy more than 100.

In [ ]:
# TODO: solve the policy model, then make the plan table
policy_plan = None

# --- Look: the policy plan ---
policy_plan.round(2)

## Step 3: Compute the cost of the policy

The cost of the policy is the margin lost by adding it: `lp_margin` minus the
policy model's total margin. Print both margins and the difference.

In [ ]:
# TODO: the cost of the policy in dollars of margin

## Step 4: Read the shadow prices

Loop over `policy.constraints.items()` and print each constraint's name, slack, and
shadow price, as in the chapter's section on shadow prices. Use `abs` on the slack,
as in the chapter, to remove the minus sign of −0.0.

In [ ]:
# TODO: slack and shadow price of each of the eight constraints

## Step 5: Policy result

Write two sentences. First, state the new plan, its margin, and the cost of the policy.
Then state how the shadow price of space and the shadow price of the budget changed
from the original LP (recap cell), and explain each change with the products that
are limited by the policy and the products that are not.

---

**New plan and cost of the policy:** *replace with your answer*

**Change in the shadow prices:** *replace with your answer*

---

## Step 6 (Evaluate lab): Re-solve at 21 budgets

The 21 budgets from \$20,000 to \$30,000 in steps of \$500 are provided in
`BUDGETS`. For each budget, build the model with `build_model(250, budget)`, solve
it, and append one row with the budget, the total margin, and the budget's shadow
price, `.constraints["budget"].pi`. Make the DataFrame `budgets` from the rows, with
the columns `budget`, `margin`, and `budget_price`. The loop has the form of the
chapter's loop over display sizes, with the budget as the input that changes.

In [ ]:
BUDGETS = list(range(20_000, 30_001, 500))     # 20,000, 20,500, ..., 30,000

# TODO: re-solve the original LP at each budget
budgets = None

# --- Look: the shape and the first rows ---
print(budgets.shape)
budgets.head().round(4)

## Step 7: Compute the margin gained per added dollar

Add the column `gain_per_dollar` to `budgets`: the margin minus the margin of the
row before, divided by 500. Use `shift(1)`, as `shift(52)` was used in Chapter 11.
The first row has no row before it, so its value is missing.

In [ ]:
# TODO: margin gained per added dollar between consecutive budgets

# --- Look: the whole table ---
budgets[["budget", "margin", "budget_price", "gain_per_dollar"]]

## Step 8: Find the range of the shadow price

Keep the rows of `budgets` whose `budget_price`, rounded to 4 decimals, equals
`budget_price` from the recap cell, rounded the same way. Print the smallest and
the largest budget among them, as in the chapter's range of the shadow price of
space.

In [ ]:
# TODO: the budgets with the same shadow price as at $25,000

## Step 9: Range of the budget's shadow price

Write two sentences. First, compare the margin gained per added dollar with the shadow
price of \$0.45 and state the range of budgets over which the shadow price held.
Then say what you would tell finance about raising the budget to \$28,000, using
the margin and the shadow price at that budget.

---

**Range of the shadow price:** *replace with your answer*

**Advice on a budget of \$28,000:** *replace with your answer*

---

## Submission

Run **Restart and run all** and confirm the notebook executes top to bottom with no
errors. Then download it as `.ipynb` and upload it to the Lab 13 dropbox.

If you used an AI assistant, add a one-line note naming the tool and what it
generated, per the course policy in Appendix D.